In [26]:
import glob
import os
import pandas as pd
import sys
import numpy as np
import subprocess
import matplotlib.pyplot as plt
import gemmi
import tarfile
import re

In [10]:
def read_ene(ene_file):
    """
    Read a pyDock energy file and return a DataFrame with the energies.
    Script to extract the energies from the pydock output files and save them in a CSV file.
    A typical output of the script is as follows:
            Conf         Ele      Desolv         VDW       Total        RANK
    ------------------------------------------------------------------------
            1     -24.689     -12.002    -102.896     -46.981           1
    This file returns a CSV file with the following columns:
    Conf, Ele, Desolv, VDW, 0.1VdW-Total, 1.0VdW-Total, RANK
    """
    columns = ["Conf", "Ele", "Desolv", "VDW", "Total", "RANK"]

    df = pd.read_csv(
        ene_file,
        sep=r"\s+",
        skiprows=2,
        header=None,
        names=columns,
        comment="#",
    )

    # Convert energy and ranking columns to numeric values
    for col in columns:
        df[col] = pd.to_numeric(df[col], errors="raise")

    # Rename the original total energy
    df.rename(columns={"Total": "0.1VdW-Total"}, inplace=True)

    # Calculate the total using the full VDW contribution
    df["1.0VdW-Total"] = (
        df["Ele"] + df["Desolv"] + df["VDW"]
    ).round(3)  # Round to 3 decimal places

    # Reorder the column 1.0VdW-Total to be after 0.1VdW-Total
    df = df[["Conf", "Ele", "Desolv", "0.1VdW-Total", "1.0VdW-Total", "RANK"]]

    return df

In [14]:
def create_pydock_inputs(
        pydock_in_file,
        receptor_pdb, receptor_mol, receptor_newmol,
        ligand_pdb, ligand_mol, ligand_newmol
):
    """
    Create pyDock4 input files for given receptor and ligand PDB files.
    mol and newmol can be either a single chain (e.g., A) or a list of 
    chains (e.g., A,B,C).
    """
    # Create the input file content
    input_content = f"""[receptor]
pdb = {receptor_pdb}
mol = {receptor_mol}
newmol = {receptor_newmol}
[ligand]
pdb = {ligand_pdb}
mol = {ligand_mol}
newmol = {ligand_newmol}
"""
    # Write the input file
    input_file_name = os.path.join(pydock_in_file)
    # input file must have suffix .ini; if not, add it
    if not input_file_name.endswith(".ini"):
        input_file_name += ".ini"
    with open(input_file_name, "w") as f:
        f.write(input_content)
    # chmod the input file to be executable
    os.chmod(input_file_name, 0o755)


In [31]:
def run_DockQ(
    model_pdb,
    reference_pdb,
    dockq_model="short",
    mapping=None,
):
    command = [
        "DockQ",
        model_pdb,
        reference_pdb,
        f"--{dockq_model}",
    ]

    if mapping is not None:
        command.append(f"--mapping={mapping}")

    result = subprocess.run(
        command,
        capture_output=True,
        text=True,
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"DockQ failed for {model_pdb} against {reference_pdb}\n"
            f"Command: {' '.join(command)}\n"
            f"stderr:\n{result.stderr}"
        )

    return parse_dockq_output(result.stdout)

def parse_dockq_output(output):
    """
    Parse DockQ output into a dictionary of total and per-interface metrics.
    """
    results = {}

    # Extract total DockQ score
    total_match = re.search(
        r"Total DockQ .*?:\s*([-+]?\d*\.?\d+)",
        output
    )
    if total_match:
        results["TotalDockQ"] = float(total_match.group(1))
    else:
        raise ValueError("Could not find Total DockQ in output.")

    # Extract metrics from each interface line
    metrics = ["DockQ", "iRMSD", "LRMSD", "fnat",
               "fnonnat", "F1", "clashes"]

    for line in output.splitlines():
        if not re.match(r"\s*DockQ\s+", line):
            continue

        mapping_match = re.search(r"\bmapping\s+(\S+)", line)
        if not mapping_match:
            continue

        interface = mapping_match.group(1).split(":")[0]

        patterns = {
            "DockQ": r"\bDockQ\s+([-+]?\d*\.?\d+)",
            "iRMSD": r"\biRMSD\s+([-+]?\d*\.?\d+)",
            "LRMSD": r"\bLRMSD\s+([-+]?\d*\.?\d+)",
            "fnat": r"\bfnat\s+([-+]?\d*\.?\d+)",
            "fnonnat": r"\bfnonnat\s+([-+]?\d*\.?\d+)",
            "F1": r"\bF1\s+([-+]?\d*\.?\d+)",
            "clashes": r"\bclashes\s+(\d+)",
        }

        for metric in metrics:
            match = re.search(patterns[metric], line)
            if match:
                value = (
                    int(match.group(1))
                    if metric == "clashes"
                    else float(match.group(1))
                )
                results[f"{metric}_{interface}"] = value

    return results


In [11]:
#### Functions
def extract_id(model_name):
    parts = model_name.split("_")
    seed = None
    model_num = None
    for i, part in enumerate(parts):
        if part == "seed":
            seed = parts[i + 1]
        elif part == "model":
            model_num = parts[i + 1]
    return seed, model_num

In [ ]:
af_models = [
    "/home/guillem/Documents/Projects/IgAse-IgA_modelling/IgAse_IgA2_wt_multiple_seeds_1",
    "/home/guillem/Documents/Projects/IgAse-IgA_modelling/IgAse_IgA2_wt_multiple_seeds_2",
    "/home/guillem/Documents/Projects/IgAse-IgA_modelling/IgAse_IgA2_wt_multiple_seeds_3",
    "/home/guillem/Documents/Projects/IgAse-IgA_modelling/IgAse_IgA2_wt_multiple_seeds_4"
]
cif_files = [
    cif for i in af_models for cif in glob.glob(
    os.path.join(i, "**/*.cif"), recursive=False)
    ]
print(f"Found {len(cif_files)} mmCIF files for analysis.")


Found 500 mmCIF files for analysis.


In [13]:
af3_metrics_df = pd.read_csv(
    "/home/guillem/Documents/Projects/IgAse-IgA_modelling/AF3S_full_scores_clusters_tmscores.csv"
)
af3_metrics_df

,model_name,seed,model_num,iptm,ptm,ranking_score,chainA_iptm,chainB_iptm,chainC_iptm,chainD_iptm,...,LIS_min_B-C,LIS_max_B-C,ipTM_af_A-B,ipTM_af_A-C,ipTM_af_B-C,cluster,TMscore_precat_0,TMscore_precat_1,TMscore_recog_0,TMscore_recog_1
0,fold_igase_iga2_wt_seed_204865_model_2,204865,2,0.64,0.73,0.65,0.77,0.57,0.53,0.63,...,0.5223,0.5355,0.60,0.55,0.68,14,0.72301,0.78238,0.71134,0.77179
1,fold_igase_iga2_wt_seed_204865_model_0,204865,0,0.64,0.73,0.66,0.77,0.58,0.53,0.63,...,0.5171,0.5323,0.61,0.56,0.68,14,0.73560,0.79597,0.72287,0.78412
2,fold_igase_iga2_wt_seed_204865_model_3,204865,3,0.63,0.73,0.65,0.77,0.57,0.53,0.63,...,0.5242,0.5354,0.60,0.55,0.68,14,0.72667,0.78635,0.72015,0.78130
3,fold_igase_iga2_wt_seed_204865_model_1,204865,1,0.64,0.73,0.66,0.77,0.58,0.53,0.63,...,0.5252,0.5361,0.61,0.56,0.68,14,0.72491,0.78454,0.71377,0.77430
4,fold_igase_iga2_wt_seed_204865_model_4,204865,4,0.63,0.72,0.65,0.76,0.56,0.52,0.62,...,0.5257,0.5433,0.59,0.54,0.68,14,0.71729,0.77633,0.71302,0.77397
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,fold_igase_iga2_wt_seed_171858_model_3,171858,3,0.63,0.73,0.65,0.76,0.56,0.51,0.61,...,0.4760,0.4843,0.58,0.53,0.71,14,0.71259,0.77140,0.70871,0.76940
496,fold_igase_iga2_wt_seed_171858_model_2,171858,2,0.65,0.74,0.67,0.77,0.57,0.53,0.62,...,0.4739,0.4846,0.60,0.55,0.72,14,0.71094,0.76970,0.70032,0.76003
497,fold_igase_iga2_wt_seed_171858_model_4,171858,4,0.63,0.71,0.65,0.77,0.56,0.51,0.63,...,0.5147,0.5150,0.61,0.55,0.63,15,0.85531,0.92435,0.83152,0.90038
498,fold_igase_iga2_wt_seed_171858_model_1,171858,1,0.65,0.74,0.67,0.77,0.58,0.53,0.63,...,0.4745,0.4854,0.60,0.56,0.71,14,0.70774,0.76636,0.69771,0.75744


In [ ]:
# Use the cif_path column in af3_metrics_df
# to iterate over the mmCIF files to create the pyDock
# input files
for index, row in af3_metrics_df.iterrows():
    cif_path = row["cif_path"]

    cif_path = cif_path.replace(
        "/home/gaecri/Documents/Projects/IgAses/IgAse_IgA2_modelling/AF3_server/outs/",
        "/home/guillem/Documents/Projects/IgAse-IgA_modelling/"
    )
    model_name = os.path.basename(cif_path).replace(".cif", "")
    seed, model_num = extract_id(model_name)
    # receptor and ligand pdb are the cif_path with the .cif replaced by .pdb
    pdb_path = cif_path.replace(".cif", ".pdb")
    receptor_pdb = pdb_path
    ligand_pdb = pdb_path
    # Create the pdb_path if it does not exist
    if not os.path.exists(pdb_path):

    # Create the pyDock input file using gemmi to read the mmCIF file and write the PDB file
        doc = gemmi.cif.read_file(cif_path)
        block = doc.sole_block()
        structure = gemmi.make_structure_from_block(block)
        structure.write_pdb(pdb_path)

    pydock_in_file = f"/home/guillem/Documents/Projects/IgAse-IgA_modelling/pydock_inputs/{model_name}.ini"
    create_pydock_inputs(
        pydock_in_file,
        receptor_pdb, "A", "A",
        ligand_pdb, "B,C", "B,C"
    )
    # print(f"Created pyDock input file: {pydock_in_file}")
    # break # Remove this break to process all rows

In [ ]:
# Now we can run the pyDock4 program using the created input files.
pydock_path = "/home/guillem/Programs/pydock4/pydock4"
conda_exe = "/home/guillem/miniconda3/bin/conda"

for pydock_in_file in glob.glob("/home/guillem/Documents/Projects/IgAse-IgA_modelling/pydock_inputs/*.ini"):
    print(f"Running pyDock4 with input file: {pydock_in_file}")
    # Run the pyDock4 program
    # pydock_in_file must not have the .ini suffix, so we remove it if it exists
    if pydock_in_file.endswith(".ini"):
        pydock_in_file_no_ini = pydock_in_file[:-4]
    # If the output file exists, we skip the run
    pydock_out_file = f"{pydock_in_file_no_ini}.ene"
    if not os.path.exists(pydock_out_file):
        subprocess.run(
            [
                conda_exe, "run", "--no-capture-output", "-n", "bioinfo",
                "/home/guillem/Programs/pydock4/pydock4",
                pydock_in_file_no_ini, "bindEy"
            ],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL,
            check=True
        )
    # .tar.gz the outputs created by pyDock4 for each input file
    # and remove the original files:
    # pydock_in_file_no_ini + [
    # _lig.pdb, _lig.pdb.amber, _lig.pdb.H,
    # _rec.pdb, _rec.pdb.amber, _rec.pdb.H, .rot]
    # Create the archive
    tar_file = f"{pydock_in_file_no_ini}.tar.gz"

    with tarfile.open(tar_file, "w:gz") as tar:
        for file in glob.glob(f"{pydock_in_file_no_ini}*"):
            # Do not add the archive to itself
            # And do not add the .ene
            if file != tar_file and not file.endswith(".ene"):
                tar.add(file, arcname=os.path.basename(file))

    # Remove original files, but preserve the archive
    # Do not remove the .ene file, as it is the output of pyDock4
    for file in glob.glob(f"{pydock_in_file_no_ini}*"):
        if file != tar_file and not file.endswith(".ene"):
            os.remove(file)
    # break # Remove this break to process all input files

In [ ]:
# Now we can read the output energy files and save them in a CSV file
# to merge with the AF3 metrics dataframe. The output energy files are in the same directory as the input files, with the same name but with the suffix .ene. We will read all the .ene files and merge them with the AF3 metrics dataframe.
energy_dfs = []
for ene_file in glob.glob("/home/guillem/Documents/Projects/IgAse-IgA_modelling/pydock_inputs/*.ene"):
    print(f"Reading energy file: {ene_file}")
    energy_df = read_ene(ene_file)
    # Add a column with the model name extracted from the ene_file
    model_name = os.path.basename(ene_file).replace(".ene", "")
    energy_df["model_name"] = model_name
    energy_dfs.append(energy_df)

# Concatenate all the energy dataframes
all_energy_df = pd.concat(energy_dfs, ignore_index=True)

# Merge the energy dataframe with the AF3 metrics dataframe on the model_name column
merged_df = pd.merge(af3_metrics_df, all_energy_df, on="model_name", how="left")

# Save the merged dataframe to a CSV file
merged_df.to_csv("/home/guillem/Documents/Projects/IgAse-IgA_modelling/AF3S_with_energies.csv", index=False)


In [33]:
# Compute DockQ for all models against two reference structures:
precat_pdb = "/home/guillem/Documents/Projects/IgAse-IgA_modelling/IgAse23-Fabpr_precat_last.pdb"
recogn_pdb = "/home/guillem/Documents/Projects/IgAse-IgA_modelling/IgAse23-Fabpr_recog_last.pdb"

merged_df = pd.read_csv(
    "/home/guillem/Documents/Projects/IgAse-IgA_modelling/AF3S_with_energies_z_scores.csv"
)

# Iterate over the models in the merged_df dataframe, and for each model, run DockQ against the two reference structures. Store the results in a new dataframe.
dockq_results = []
for index, row in merged_df.iterrows():

    cif_path = row["cif_path"]
    cif_path = cif_path.replace(
        "/home/gaecri/Documents/Projects/IgAses/IgAse_IgA2_modelling/AF3_server/outs/",
        "/home/guillem/Documents/Projects/IgAse-IgA_modelling/"
    )
    model_name = os.path.basename(cif_path).replace(".cif", "")
    model_pdb = cif_path.replace(".cif", ".pdb")
    # print(f"Processing model: {model_name}")

    # Run DockQ against the precat and recogn reference structures
    dockq_precat = run_DockQ(
        model_pdb, precat_pdb,
        dockq_model="short",
        mapping="ABC:EHL",
        )
    dockq_recogn = run_DockQ(
        model_pdb, recogn_pdb,
        dockq_model="short",
        mapping="ABC:EHL",
        )
    # Combine the results into a single dictionary
    combined_results = {
        "model_name": model_name,
        **{f"precat_{k}": v for k, v in dockq_precat.items()},
        **{f"recogn_{k}": v for k, v in dockq_recogn.items()},
    }
    dockq_results.append(combined_results)
    # print("Results:", combined_results)
    # break # Remove this break to process all models

# Create a dataframe from the dockq_results list of dictionaries
dockq_df = pd.DataFrame(dockq_results)
# dockq_df

# Merge the dockq_df with the merged_df on the model_name column
final_df = pd.merge(merged_df, dockq_df, on="model_name", how="left")
final_df

,model_name,seed,model_num,iptm,ptm,ranking_score,chainA_iptm,chainB_iptm,chainC_iptm,chainD_iptm,...,recogn_fnonnat_AC,recogn_F1_AC,recogn_clashes_AC,recogn_DockQ_BC,recogn_iRMSD_BC,recogn_LRMSD_BC,recogn_fnat_BC,recogn_fnonnat_BC,recogn_F1_BC,recogn_clashes_BC
0,fold_igase_iga2_wt_seed_204865_model_2,204865,2,0.64,0.73,0.65,0.77,0.57,0.53,0.63,...,0.455,0.480,0,0.435,6.269,10.422,0.852,0.194,0.829,0
1,fold_igase_iga2_wt_seed_204865_model_0,204865,0,0.64,0.73,0.66,0.77,0.58,0.53,0.63,...,0.455,0.480,0,0.457,5.506,9.236,0.844,0.214,0.814,0
2,fold_igase_iga2_wt_seed_204865_model_3,204865,3,0.63,0.73,0.65,0.77,0.57,0.53,0.63,...,0.538,0.444,0,0.455,6.002,10.110,0.893,0.193,0.848,0
3,fold_igase_iga2_wt_seed_204865_model_1,204865,1,0.64,0.73,0.66,0.77,0.58,0.53,0.63,...,0.500,0.462,0,0.432,6.393,10.627,0.852,0.212,0.819,0
4,fold_igase_iga2_wt_seed_204865_model_4,204865,4,0.63,0.72,0.65,0.76,0.56,0.52,0.62,...,0.500,0.462,0,0.423,6.773,11.289,0.861,0.198,0.830,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
495,fold_igase_iga2_wt_seed_171858_model_3,171858,3,0.63,0.73,0.65,0.76,0.56,0.51,0.61,...,0.500,0.462,0,0.418,6.896,11.646,0.861,0.173,0.843,0
496,fold_igase_iga2_wt_seed_171858_model_2,171858,2,0.65,0.74,0.67,0.77,0.57,0.53,0.62,...,0.538,0.444,0,0.419,6.951,11.744,0.869,0.185,0.841,0
497,fold_igase_iga2_wt_seed_171858_model_4,171858,4,0.63,0.71,0.65,0.77,0.56,0.51,0.63,...,0.500,0.462,0,0.664,2.278,3.520,0.836,0.215,0.810,0
498,fold_igase_iga2_wt_seed_171858_model_1,171858,1,0.65,0.74,0.67,0.77,0.58,0.53,0.63,...,0.538,0.444,0,0.407,7.139,11.995,0.844,0.195,0.824,0


In [35]:
final_df.to_csv(
    "/home/guillem/Documents/Projects/IgAse-IgA_modelling/AF3S_with_DockQ.csv",
    index=False
)